# 🐍 Master Guide: Descriptors & Attribute Lookup Mechanics
> **Single-Source Revision Guide**: Covers the descriptor protocol (`__get__`, `__set__`, `__set_name__`), Data vs Non-Data descriptors, and `__getattr__` vs `__getattribute__`.
>
> *Run each cell interactively with `Shift + Enter`.*

---

## 📌 1. Topic Overview
Descriptors power almost every major feature in Python OOP: `@property`, `@classmethod`, `@staticmethod`, methods, and ORM fields (Django/SQLAlchemy). A descriptor is an object attribute with binding behavior defined by the descriptor protocol.

---
## 🧠 2. CPython Architecture: Data vs Non-Data Descriptors

- **Data Descriptor**: Implements `__set__()` (and optionally `__get__()`). Overrides instance `__dict__`!
- **Non-Data Descriptor**: Implements ONLY `__get__()` (e.g. methods). Yields priority to instance `__dict__`!

```
Exact Priority Sequence for obj.attr:
  1. Class & base classes searched for DATA DESCRIPTORS (has __set__)
     -> If found: Call desc.__get__(obj, type(obj))
  2. Instance dictionary obj.__dict__['attr']
     -> If found: Return value directly
  3. Class & base classes searched for NON-DATA DESCRIPTORS (only __get__)
     -> If found: Call desc.__get__(obj, type(obj))
  4. Class & base classes searched for standard attributes
  5. Fallback to obj.__getattr__('attr') if defined; else raise AttributeError
```

---
## 🟢 3. Part 1: Beginner Fundamentals

### Step 1: Building a Type-Validating Descriptor with `__set_name__`
`__set_name__` (Python 3.6+) automatically captures the attribute name assigned in the owner class!

In [ ]:
class PositiveInteger:
    def __set_name__(self, owner, name):
        self.private_name = f"_{name}"
    
    def __get__(self, obj, objtype=None):
        if obj is None:
            return self  # Accessed via class
        return getattr(obj, self.private_name, 0)
    
    def __set__(self, obj, value):
        if not isinstance(value, int) or value < 0:
            raise ValueError(f"Attribute must be a non-negative integer, got {value}!")
        setattr(obj, self.private_name, value)

class InventoryItem:
    # Reusable descriptors across multiple attributes:
    price = PositiveInteger()
    stock = PositiveInteger()
    
    def __init__(self, price, stock):
        self.price = price
        self.stock = stock

item = InventoryItem(100, 25)
print(f"Item: price={item.price}, stock={item.stock}")

try:
    item.price = -50
except ValueError as e:
    print("Validation caught by descriptor:", e)

---
## 🟡 4. Part 2: Intermediate Features

### `__getattr__` vs `__getattribute__`
- `__getattr__(name)`: **Fallback only**. Called ONLY when `name` was NOT found anywhere in `__dict__` or class MRO.
- `__getattribute__(name)`: **Intercepts EVERY access**, even if the attribute exists! (Risk of infinite recursion).

In [ ]:
class DynamicProxy:
    def __init__(self):
        self.existing = "I exist!"
    
    def __getattr__(self, name):
        # Called only for missing attributes:
        return f"Synthetic attribute '{name}' generated on the fly!"

proxy = DynamicProxy()
print("Existing attribute:", proxy.existing)
print("Missing attribute :", proxy.any_random_field)

---
## 📋 5. Part 3: Quick Revision Cheat Sheet

```
+------------------------------------+---------------------------------------+
| Concept                            | Key Rule                              |
+------------------------------------+---------------------------------------+
| Data descriptor                    | Has __set__; beats instance __dict__  |
| Non-data descriptor                | Only has __get__; beaten by __dict__  |
| __set_name__(owner, name)          | Captures attribute name automatically |
| __getattr__(name)                  | Called ONLY when attribute is missing |
| __getattribute__(name)             | Called on EVERY access (danger loop!) |
+------------------------------------+---------------------------------------+
```

---
## 🎓 Practical Hands-On & Real-World Examples

### 1. Enterprise Field Validation Descriptors (ORM Simulation)
> **Core Principle**:
> - Instead of rewriting repetitive getters, setters, and validation checks across dozens of model classes, encapsulate the validation logic inside reusable descriptor classes.
> - `StringField` enforces string typing and minimum length.
> - `IntegerField` enforces range constraints (e.g. marks between 0 and 100).

In [ ]:
# Reusable ORM-Style Descriptors
class ValidatedString:
    def __init__(self, min_length=1):
        self.min_length = min_length

    def __set_name__(self, owner, name):
        self.storage_name = f"_{name}"

    def __get__(self, instance, owner=None):
        if instance is None:
            return self
        return getattr(instance, self.storage_name, "")

    def __set__(self, instance, value):
        if not isinstance(value, str) or len(value.strip()) < self.min_length:
            raise ValueError(f"Attribute must be a string with minimum {self.min_length} chars!")
        setattr(instance, self.storage_name, value.strip())

class ValidatedScore:
    def __init__(self, min_val=0.0, max_val=100.0):
        self.min_val = min_val
        self.max_val = max_val

    def __set_name__(self, owner, name):
        self.storage_name = f"_{name}"

    def __get__(self, instance, owner=None):
        if instance is None:
            return self
        return getattr(instance, self.storage_name, 0.0)

    def __set__(self, instance, value):
        if not isinstance(value, (int, float)) or not (self.min_val <= value <= self.max_val):
            raise ValueError(f"Score must be a number between {self.min_val} and {self.max_val}!")
        setattr(instance, self.storage_name, float(value))

class StudentEvaluation:
    student_name = ValidatedString(min_length=3)
    exam_score = ValidatedScore(min_val=0.0, max_val=100.0)

    def __init__(self, name, score):
        self.student_name = name
        self.exam_score = score

eval1 = StudentEvaluation("Nandini", 95.5)
print(f"Valid Record: Name='{eval1.student_name}', Score={eval1.exam_score}")

for invalid_val in [("", 85), ("Rohit", 150)]:
    try:
        StudentEvaluation(*invalid_val)
    except ValueError as err:
        print(f"Caught Invalid Input {invalid_val}: {err}")

### 2. Lazy Cached Property via Non-Data Descriptor
A non-data descriptor executes an expensive computation once, saves the result into `instance.__dict__`, and automatically steps out of the way for future lookups.

In [ ]:
# Lazy Cached Property Implementation
import time

class LazyCachedProperty:
    def __init__(self, func):
        self.func = func
        self.name = func.__name__

    def __get__(self, instance, owner=None):
        if instance is None:
            return self
        print(f"  [Computing...] Executing expensive calculation for '{self.name}'")
        value = self.func(instance)
        # Overwrite the instance __dict__ key so future lookups bypass __get__ directly!
        instance.__dict__[self.name] = value
        return value

class DatasetAnalytics:
    def __init__(self, record_count):
        self.record_count = record_count

    @LazyCachedProperty
    def heavy_aggregate(self):
        time.sleep(0.05)  # Simulate expensive database aggregation
        return sum(i for i in range(self.record_count))

data = DatasetAnalytics(100_000)
print("Access 1:", data.heavy_aggregate)  # Executes descriptor __get__
print("Access 2:", data.heavy_aggregate)  # Fetched directly from __dict__ with zero latency!
print("Cached in __dict__:", data.__dict__)